# Module 3: Walk-Forward Validation with GA Feature Selection

## Learning Objectives

By completing this notebook, you will:
1. Understand the importance of temporal validation in time series
2. Implement walk-forward validation for GA-based feature selection
3. Prevent data leakage and lookahead bias in time series models
4. Evaluate feature stability across different time windows
5. Apply GAs to time series forecasting with appropriate validation

## Prerequisites

- Module 1 completed (GA fundamentals)
- Module 2 completed (fitness functions)
- Understanding of time series concepts (stationarity, autocorrelation)
- DEAP library installed (`pip install deap`)

## Estimated Time: 75 minutes

---

## 1. Why Walk-Forward Validation?

### Key Concept: Time series data has temporal dependencies that violate standard cross-validation assumptions.

**Problems with Standard K-Fold CV:**
1. **Lookahead bias**: Training on future data, testing on past
2. **Data leakage**: Shuffling destroys temporal structure
3. **Overfitting**: Model learns future information

**Walk-Forward Validation:**
- Train on historical data only
- Test on immediate future
- Progressively move forward in time
- Mimics real-world deployment

**Process:**
```
Window 1: [Train----] [Test]
Window 2:   [Train----] [Test]
Window 3:     [Train----] [Test]
```

In [ ]:
# Import required libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import warnings
warnings.filterwarnings('ignore')

# DEAP imports
from deap import base, creator, tools, algorithms
import random

# Set random seeds
np.random.seed(42)
random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("All libraries imported successfully!")

### 1.1 Create Synthetic Time Series Dataset

Generate realistic time series with trend, seasonality, and noise.

In [ ]:
# Purpose: Generate synthetic time series with multiple features
# Key Concept: Include relevant and irrelevant features to test GA

def generate_time_series_data(n_samples=1000, n_relevant=5, n_irrelevant=15, noise_level=0.1):
    """
    Generate synthetic time series data for feature selection.
    
    Parameters
    ----------
    n_samples : int
        Number of time steps
    n_relevant : int
        Number of relevant features
    n_irrelevant : int
        Number of irrelevant features
    noise_level : float
        Standard deviation of noise
    
    Returns
    -------
    df : pd.DataFrame
        Time series dataframe with features and target
    """
    # Time index
    time = np.arange(n_samples)
    
    # Generate relevant features
    relevant_features = {}
    
    # Feature 1: Linear trend
    relevant_features['trend'] = 0.5 * time / n_samples
    
    # Feature 2: Annual seasonality
    relevant_features['seasonal_annual'] = np.sin(2 * np.pi * time / 365)
    
    # Feature 3: Monthly seasonality
    relevant_features['seasonal_monthly'] = np.cos(2 * np.pi * time / 30)
    
    # Feature 4: Autoregressive component
    ar_component = np.zeros(n_samples)
    ar_component[0] = np.random.randn()
    for t in range(1, n_samples):
        ar_component[t] = 0.7 * ar_component[t-1] + np.random.randn() * 0.1
    relevant_features['ar_component'] = ar_component
    
    # Feature 5: Weekly pattern
    relevant_features['weekly_pattern'] = np.sin(2 * np.pi * time / 7)
    
    # Generate irrelevant features (random noise)
    irrelevant_features = {}
    for i in range(n_irrelevant):
        irrelevant_features[f'noise_{i+1}'] = np.random.randn(n_samples)
    
    # Create target as weighted sum of relevant features + noise
    target = (
        2.0 * relevant_features['trend'] +
        1.5 * relevant_features['seasonal_annual'] +
        1.0 * relevant_features['seasonal_monthly'] +
        0.8 * relevant_features['ar_component'] +
        0.5 * relevant_features['weekly_pattern'] +
        np.random.randn(n_samples) * noise_level
    )
    
    # Combine into dataframe
    data = {**relevant_features, **irrelevant_features}
    df = pd.DataFrame(data)
    df['target'] = target
    df.index = pd.date_range('2020-01-01', periods=n_samples, freq='D')
    
    return df

# Generate data
df = generate_time_series_data(n_samples=1000, n_relevant=5, n_irrelevant=15, noise_level=0.1)

print(f"Dataset shape: {df.shape}")
print(f"\nFeatures:")
print(df.columns.tolist())
print(f"\nFirst few rows:")
print(df.head())

### 1.2 Visualize Time Series

In [ ]:
# Purpose: Visualize target and key features
# Key Concept: Understand temporal patterns in data

fig, axes = plt.subplots(3, 2, figsize=(15, 10))

# Plot target
axes[0, 0].plot(df.index, df['target'], linewidth=1.5, color='black')
axes[0, 0].set_title('Target Variable', fontsize=12)
axes[0, 0].set_ylabel('Value', fontsize=10)
axes[0, 0].grid(alpha=0.3)

# Plot relevant features
relevant_cols = ['trend', 'seasonal_annual', 'seasonal_monthly', 'ar_component', 'weekly_pattern']
for i, col in enumerate(relevant_cols, 1):
    row = i // 2
    col_idx = i % 2
    axes[row, col_idx].plot(df.index, df[col], linewidth=1.5)
    axes[row, col_idx].set_title(f'{col}', fontsize=12)
    axes[row, col_idx].set_ylabel('Value', fontsize=10)
    axes[row, col_idx].grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("Time series components visualized.")
print("Note: Target is influenced by all 5 relevant features shown above.")

## 2. Walk-Forward Validation Implementation

### Key Concept: Train on past, test on immediate future, then move forward.

In [ ]:
# Purpose: Implement walk-forward validation split
# Key Concept: Sliding window that respects temporal order

class WalkForwardValidation:
    """
    Walk-forward validation for time series.
    """
    
    def __init__(self, n_splits=5, test_size=100):
        """
        Parameters
        ----------
        n_splits : int
            Number of splits
        test_size : int
            Number of samples in test set
        """
        self.n_splits = n_splits
        self.test_size = test_size
    
    def split(self, X, y=None):
        """
        Generate train/test indices.
        
        Yields
        ------
        train_idx : np.ndarray
            Training indices
        test_idx : np.ndarray
            Test indices
        """
        n_samples = len(X)
        
        # Calculate minimum training size
        min_train_size = (n_samples - self.n_splits * self.test_size) // self.n_splits
        
        for i in range(self.n_splits):
            # Training set grows with each split
            train_end = min_train_size + i * self.test_size + (i * min_train_size // self.n_splits)
            test_start = train_end
            test_end = test_start + self.test_size
            
            if test_end > n_samples:
                break
            
            train_idx = np.arange(0, train_end)
            test_idx = np.arange(test_start, test_end)
            
            yield train_idx, test_idx
    
    def get_n_splits(self, X=None, y=None, groups=None):
        """Return number of splits."""
        return self.n_splits

# Test walk-forward validation
wfv = WalkForwardValidation(n_splits=5, test_size=100)

print("Walk-Forward Validation Splits:")
print("="*70)

for i, (train_idx, test_idx) in enumerate(wfv.split(df), 1):
    print(f"\nSplit {i}:")
    print(f"  Train: samples {train_idx[0]:4d} to {train_idx[-1]:4d} ({len(train_idx):4d} samples)")
    print(f"  Test:  samples {test_idx[0]:4d} to {test_idx[-1]:4d} ({len(test_idx):4d} samples)")

### 2.1 Visualize Walk-Forward Splits

In [ ]:
# Purpose: Visualize train/test splits over time
# Key Concept: Each split uses only past data for training

fig, ax = plt.subplots(figsize=(14, 6))

# Plot each split
wfv = WalkForwardValidation(n_splits=5, test_size=100)
colors = plt.cm.Set3(np.linspace(0, 1, 5))

for i, (train_idx, test_idx) in enumerate(wfv.split(df)):
    # Plot training period
    ax.barh(i, len(train_idx), left=train_idx[0], height=0.8,
            color=colors[i], alpha=0.6, label=f'Split {i+1} Train' if i == 0 else '')
    
    # Plot test period
    ax.barh(i, len(test_idx), left=test_idx[0], height=0.8,
            color=colors[i], alpha=1.0, edgecolor='black', linewidth=2,
            label=f'Split {i+1} Test' if i == 0 else '')

ax.set_xlabel('Sample Index', fontsize=12)
ax.set_ylabel('Split Number', fontsize=12)
ax.set_title('Walk-Forward Validation Splits', fontsize=14)
ax.set_yticks(range(5))
ax.set_yticklabels([f'Split {i+1}' for i in range(5)])
ax.grid(alpha=0.3, axis='x')

# Add legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='gray', alpha=0.6, label='Training Period'),
    Patch(facecolor='gray', alpha=1.0, edgecolor='black', linewidth=2, label='Test Period')
]
ax.legend(handles=legend_elements, loc='upper left', fontsize=11)

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Light bars: Training periods (growing with each split)")
print("  - Dark bars with borders: Test periods")
print("  - No overlap between train and test within each split")

## 3. GA Feature Selection with Walk-Forward Validation

### Key Concept: Fitness function must use walk-forward validation to avoid lookahead bias.

In [ ]:
# Purpose: Setup DEAP for time series feature selection
# Key Concept: Fitness evaluates performance across multiple walk-forward windows

# Prepare data
X = df.drop('target', axis=1)
y = df['target']

# Clean up existing types
if hasattr(creator, "FitnessMax"):
    del creator.FitnessMax
if hasattr(creator, "Individual"):
    del creator.Individual

# Create fitness and individual types
creator.create("FitnessMax", base.Fitness, weights=(1.0,))
creator.create("Individual", list, fitness=creator.FitnessMax)

print("DEAP types created for time series feature selection.")

In [ ]:
# Purpose: Define fitness function with walk-forward validation
# Key Concept: Average performance across all walk-forward splits

def evaluate_time_series_features(individual, X_data, y_data, model_class=Ridge):
    """
    Evaluate feature subset using walk-forward validation.
    
    Parameters
    ----------
    individual : list
        Binary chromosome
    X_data : pd.DataFrame
        Feature data
    y_data : pd.Series
        Target data
    model_class : class
        Scikit-learn model class
    
    Returns
    -------
    fitness : tuple
        Negative RMSE (for maximization)
    """
    # Check for at least one feature
    if sum(individual) == 0:
        return (-999.0,)
    
    # Select features
    feature_mask = np.array(individual, dtype=bool)
    X_selected = X_data.iloc[:, feature_mask]
    
    # Walk-forward validation
    wfv = WalkForwardValidation(n_splits=5, test_size=100)
    
    rmse_scores = []
    
    for train_idx, test_idx in wfv.split(X_selected):
        # Split data
        X_train = X_selected.iloc[train_idx]
        X_test = X_selected.iloc[test_idx]
        y_train = y_data.iloc[train_idx]
        y_test = y_data.iloc[test_idx]
        
        # Scale data
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Train model
        model = model_class(random_state=42)
        model.fit(X_train_scaled, y_train)
        
        # Predict and evaluate
        y_pred = model.predict(X_test_scaled)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        rmse_scores.append(rmse)
    
    # Average RMSE across all splits
    avg_rmse = np.mean(rmse_scores)
    
    # Apply parsimony penalty
    n_features = sum(individual)
    parsimony_penalty = 0.01 * (n_features / len(individual))
    
    # Fitness: negative RMSE (to maximize) minus parsimony
    fitness = -avg_rmse - parsimony_penalty
    
    return (fitness,)

# Test fitness function
test_ind = [1] * 5 + [0] * 15  # Select only first 5 features (relevant ones)
test_fitness = evaluate_time_series_features(test_ind, X, y)
print(f"Test fitness (relevant features): {test_fitness[0]:.4f}")

test_ind_noise = [0] * 5 + [1] * 15  # Select only noise features
test_fitness_noise = evaluate_time_series_features(test_ind_noise, X, y)
print(f"Test fitness (noise features): {test_fitness_noise[0]:.4f}")

print("\nAs expected, relevant features have better fitness!")

### 3.1 Setup and Run GA

In [ ]:
# Purpose: Register toolbox and run GA
# Key Concept: GA finds features that perform well across all time windows

# Create toolbox
toolbox = base.Toolbox()

# Register components
toolbox.register("attr_bool", random.randint, 0, 1)
toolbox.register(
    "individual",
    tools.initRepeat,
    creator.Individual,
    toolbox.attr_bool,
    n=X.shape[1]
)
toolbox.register("population", tools.initRepeat, list, toolbox.individual)
toolbox.register("evaluate", evaluate_time_series_features, X_data=X, y_data=y)
toolbox.register("mate", tools.cxTwoPoint)
toolbox.register("mutate", tools.mutFlipBit, indpb=0.05)
toolbox.register("select", tools.selTournament, tournsize=3)

# Statistics and hall of fame
stats = tools.Statistics(key=lambda ind: ind.fitness.values)
stats.register("avg", np.mean)
stats.register("max", np.max)
stats.register("min", np.min)

hall_of_fame = tools.HallOfFame(maxsize=5)

# GA parameters
POPULATION_SIZE = 40
MAX_GENERATIONS = 30
P_CROSSOVER = 0.7
P_MUTATION = 0.2

print("Running GA with walk-forward validation...")
print(f"  Population: {POPULATION_SIZE}")
print(f"  Generations: {MAX_GENERATIONS}")
print(f"  Note: Each fitness evaluation uses 5 walk-forward splits\n")

# Create population
population = toolbox.population(n=POPULATION_SIZE)

# Run evolution
population, logbook = algorithms.eaSimple(
    population,
    toolbox,
    cxpb=P_CROSSOVER,
    mutpb=P_MUTATION,
    ngen=MAX_GENERATIONS,
    stats=stats,
    halloffame=hall_of_fame,
    verbose=True
)

print("\nGA Evolution completed!")

### 3.2 Analyze Results

In [ ]:
# Purpose: Examine selected features
# Key Concept: GA should identify the 5 relevant features

print("Top 3 Solutions from Hall of Fame:")
print("="*70)

for i, ind in enumerate(hall_of_fame[:3], 1):
    selected_features = X.columns[np.array(ind, dtype=bool)].tolist()
    fitness = ind.fitness.values[0]
    n_features = sum(ind)
    
    print(f"\nSolution {i}:")
    print(f"  Fitness: {fitness:.4f} (negative RMSE)")
    print(f"  Features: {n_features}/{X.shape[1]}")
    print(f"  Selected: {selected_features}")

# Check if relevant features were found
best_ind = hall_of_fame[0]
best_features = X.columns[np.array(best_ind, dtype=bool)].tolist()
relevant_features = ['trend', 'seasonal_annual', 'seasonal_monthly', 'ar_component', 'weekly_pattern']

found_relevant = [f for f in best_features if f in relevant_features]
print(f"\n{'='*70}")
print(f"Relevant features found: {len(found_relevant)}/{len(relevant_features)}")
print(f"Features: {found_relevant}")

### 3.3 Visualize Evolution and Results

In [ ]:
# Purpose: Visualize GA performance
# Key Concept: Monitor convergence and feature selection

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Extract logbook data
gen = logbook.select("gen")
fit_max = logbook.select("max")
fit_avg = logbook.select("avg")
fit_min = logbook.select("min")

# Plot 1: Fitness evolution
axes[0, 0].plot(gen, fit_max, 'r-', linewidth=2, label='Best')
axes[0, 0].plot(gen, fit_avg, 'b-', linewidth=2, label='Average')
axes[0, 0].plot(gen, fit_min, 'g-', linewidth=2, label='Worst')
axes[0, 0].set_xlabel('Generation', fontsize=11)
axes[0, 0].set_ylabel('Fitness (Negative RMSE)', fontsize=11)
axes[0, 0].set_title('Fitness Evolution with Walk-Forward Validation', fontsize=12)
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Plot 2: Best solution features
feature_names = [name[:15] + '...' if len(name) > 15 else name for name in X.columns]
colors_features = ['green' if X.columns[i] in relevant_features else 'gray' 
                   for i in range(len(best_ind))]
axes[0, 1].bar(range(len(best_ind)), best_ind, color=colors_features, edgecolor='black')
axes[0, 1].set_xlabel('Feature Index', fontsize=11)
axes[0, 1].set_ylabel('Selected (1) or Not (0)', fontsize=11)
axes[0, 1].set_title(f'Best Solution Features (Green = Relevant)', fontsize=12)
axes[0, 1].set_ylim(-0.1, 1.1)
axes[0, 1].grid(alpha=0.3, axis='y')

# Plot 3: Feature selection frequency across top solutions
feature_counts = np.zeros(X.shape[1])
for ind in hall_of_fame:
    feature_counts += np.array(ind)
feature_counts /= len(hall_of_fame)

sorted_idx = np.argsort(feature_counts)[::-1]
top_10_idx = sorted_idx[:10]
top_10_names = [X.columns[i] for i in top_10_idx]
top_10_counts = feature_counts[top_10_idx]

axes[1, 0].barh(range(len(top_10_names)), top_10_counts, color='steelblue', edgecolor='black')
axes[1, 0].set_yticks(range(len(top_10_names)))
axes[1, 0].set_yticklabels(top_10_names, fontsize=9)
axes[1, 0].set_xlabel('Selection Frequency', fontsize=11)
axes[1, 0].set_title('Top 10 Most Frequently Selected Features', fontsize=12)
axes[1, 0].grid(alpha=0.3, axis='x')
axes[1, 0].invert_yaxis()

# Plot 4: Number of features over generations
n_features_gen = []
for ind in population:
    n_features_gen.append(sum(ind))
axes[1, 1].hist(n_features_gen, bins=15, color='orange', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(sum(best_ind), color='red', linestyle='--', linewidth=2,
                   label=f'Best: {sum(best_ind)} features')
axes[1, 1].set_xlabel('Number of Features', fontsize=11)
axes[1, 1].set_ylabel('Frequency', fontsize=11)
axes[1, 1].set_title('Feature Count Distribution in Final Population', fontsize=12)
axes[1, 1].legend()
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Feature Stability Analysis

### Key Concept: Evaluate how consistent feature selection is across different time windows.

In [ ]:
# Purpose: Analyze feature stability across walk-forward splits
# Key Concept: Stable features are selected consistently across time

def analyze_feature_stability(individual, X_data, y_data):
    """
    Analyze which features are important in each time window.
    
    Returns
    -------
    feature_importance_by_split : list of dict
        Feature importances for each split
    """
    # Select features
    feature_mask = np.array(individual, dtype=bool)
    X_selected = X_data.iloc[:, feature_mask]
    selected_feature_names = X_data.columns[feature_mask].tolist()
    
    # Walk-forward validation
    wfv = WalkForwardValidation(n_splits=5, test_size=100)
    
    feature_importance_by_split = []
    
    for split_idx, (train_idx, test_idx) in enumerate(wfv.split(X_selected)):
        # Split and scale
        X_train = X_selected.iloc[train_idx]
        y_train = y_data.iloc[train_idx]
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        
        # Train random forest to get feature importances
        model = RandomForestRegressor(n_estimators=50, random_state=42)
        model.fit(X_train_scaled, y_train)
        
        # Store importances
        importances = dict(zip(selected_feature_names, model.feature_importances_))
        feature_importance_by_split.append(importances)
    
    return feature_importance_by_split, selected_feature_names

# Analyze best solution
best_individual = hall_of_fame[0]
importance_by_split, selected_names = analyze_feature_stability(best_individual, X, y)

print("Feature Importance Across Time Windows:")
print("="*70)

# Create dataframe
importance_df = pd.DataFrame(importance_by_split)
importance_df.index = [f"Split {i+1}" for i in range(len(importance_by_split))]

print(importance_df)

# Calculate stability (standard deviation across splits)
stability = importance_df.std(axis=0).sort_values()
print(f"\nFeature Stability (lower std = more stable):")
print(stability)

### 4.1 Visualize Feature Stability

In [ ]:
# Purpose: Visualize how feature importance changes over time
# Key Concept: Stable features maintain importance across splits

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Heatmap of importances
sns.heatmap(importance_df.T, annot=True, fmt='.3f', cmap='YlOrRd',
            cbar_kws={'label': 'Importance'}, ax=axes[0])
axes[0].set_xlabel('Time Window (Split)', fontsize=11)
axes[0].set_ylabel('Feature', fontsize=11)
axes[0].set_title('Feature Importance Across Time Windows', fontsize=12)

# Plot 2: Stability (std dev) of each feature
stability_sorted = stability.sort_values(ascending=True)
colors_stability = ['green' if feat in relevant_features else 'gray' 
                   for feat in stability_sorted.index]

axes[1].barh(range(len(stability_sorted)), stability_sorted.values,
            color=colors_stability, edgecolor='black')
axes[1].set_yticks(range(len(stability_sorted)))
axes[1].set_yticklabels(stability_sorted.index, fontsize=9)
axes[1].set_xlabel('Stability (Std Dev of Importance)', fontsize=11)
axes[1].set_title('Feature Stability (Lower = More Stable)', fontsize=12)
axes[1].invert_yaxis()
axes[1].grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

print("\nInterpretation:")
print("  - Heatmap shows how importance varies across time windows")
print("  - Bar chart shows overall stability (lower std = more consistent)")
print("  - Green bars indicate relevant features")

## 5. Exercises

### Exercise 5.1: Expanding vs Sliding Window

**Task**: Modify the `WalkForwardValidation` class to support both expanding windows (current behavior) and sliding windows (fixed-size training set that moves forward).

**Expected Output**: Comparison of feature selection with both methods.

<details>
<summary>Hint 1</summary>
Add a `window_type` parameter ('expanding' or 'sliding') and `train_size` parameter.
</details>

<details>
<summary>Hint 2</summary>
For sliding window, use `train_idx = np.arange(test_start - train_size, test_start)`.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

class WalkForwardValidationFlexible:
    """Walk-forward validation with expanding or sliding windows."""
    
    def __init__(self, n_splits=5, test_size=100, window_type='expanding', train_size=None):
        # TODO: Implement
        pass
    
    def split(self, X, y=None):
        # TODO: Implement
        pass

# Test both window types
# wfv_expanding = WalkForwardValidationFlexible(n_splits=5, window_type='expanding')
# wfv_sliding = WalkForwardValidationFlexible(n_splits=5, window_type='sliding', train_size=300)

In [ ]:
# AUTO-GRADED TEST
# -----------------
def test_exercise_5_1():
    """Test flexible walk-forward validation."""
    try:
        wfv = WalkForwardValidationFlexible(n_splits=3, test_size=50, window_type='sliding', train_size=200)
        splits = list(wfv.split(df))
        
        # Check number of splits
        assert len(splits) == 3, f"Expected 3 splits, got {len(splits)}"
        
        # Check sliding window train size
        for train_idx, test_idx in splits:
            assert len(train_idx) == 200, f"Expected train size 200, got {len(train_idx)}"
            assert len(test_idx) == 50, f"Expected test size 50, got {len(test_idx)}"
        
        print("✅ Exercise 5.1 passed!")
    except NameError:
        print("❌ WalkForwardValidationFlexible not defined")
    except Exception as e:
        print(f"❌ Test failed: {e}")

# Uncomment to test:
# test_exercise_5_1()

### Exercise 5.2: Multi-Step Ahead Forecasting

**Task**: Modify the fitness function to evaluate performance on multi-step ahead forecasts (e.g., predict 1, 3, and 7 steps ahead) and average the errors.

**Expected Output**: Feature selection optimized for multi-horizon forecasting.

<details>
<summary>Hint 1</summary>
Create lagged targets: `y_lag1 = y.shift(-1)`, `y_lag3 = y.shift(-3)`, etc.
</details>

<details>
<summary>Hint 2</summary>
Train separate models for each horizon and average their RMSEs.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

def evaluate_multistep_forecast(individual, X_data, y_data, horizons=[1, 3, 7]):
    """
    Evaluate features for multi-step ahead forecasting.
    
    Parameters
    ----------
    horizons : list
        Forecast horizons to evaluate
    """
    # TODO: Implement
    pass

# Test function
# test_fitness = evaluate_multistep_forecast(test_ind, X, y)

### Exercise 5.3: Feature Selection with Ensemble Models

**Task**: Modify the fitness function to use an ensemble of different models (Ridge, Lasso, RandomForest) and average their performance. This tests feature robustness across different model types.

**Expected Output**: Features selected work well with multiple model types.

<details>
<summary>Hint</summary>
Create a list of model classes, train each one, and average the RMSEs.
</details>

In [ ]:
# YOUR CODE HERE
# ---------------

from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor

def evaluate_ensemble_models(individual, X_data, y_data, model_classes=None):
    """
    Evaluate features using multiple model types.
    """
    if model_classes is None:
        model_classes = [Ridge, Lasso, RandomForestRegressor]
    
    # TODO: Implement
    pass

## 6. Summary

### Key Takeaways

1. **Walk-Forward Validation**: Essential for time series to prevent lookahead bias
2. **Temporal Dependencies**: Standard k-fold CV is invalid for time series data
3. **GA Fitness Function**: Must incorporate proper time series validation
4. **Feature Stability**: Evaluate consistency of feature importance across time windows
5. **Expanding vs Sliding**: Different window strategies for different use cases
6. **Production Readiness**: Walk-forward validation mimics real deployment

### Walk-Forward Best Practices

- **Never train on future data**: Always maintain temporal order
- **Multiple splits**: Use several walk-forward windows to average performance
- **Window size**: Balance between training data amount and recency
- **Feature stability**: Prefer features that remain important across time
- **Realistic testing**: Test period should match deployment frequency
- **Computational cost**: Walk-forward is expensive, optimize GA accordingly

### Common Pitfalls

- Using k-fold CV on time series (data leakage!)
- Training window too small (insufficient learning)
- Test window too large (unrealistic forecasting horizon)
- Ignoring non-stationarity (features may change importance over time)
- Not validating on truly held-out recent data

### What's Next

- **Next notebook**: Lag feature selection with GA
- **Advanced topics**: Online learning, concept drift detection
- **Real-world application**: Stock price prediction, demand forecasting

### Additional Resources

- **Time Series CV**: Bergmeir & Benítez (2012) - "On the use of cross-validation"
- **Walk-Forward Analysis**: Pardo (2008) - "The Evaluation and Optimization of Trading Strategies"
- **Feature Selection**: Chandrashekar & Sahin (2014) - "A survey on feature selection methods"